# Part 2 — Step 9: Evaluate on DermNet

Evaluates the trained EfficientNet-B0 classifier on the DermNet test set.

**Class mapping:** `Acne and Rosacea Photos` → acne (1), all other 22 conditions → non-acne (0)

**Note on class imbalance:** DermNet test set is 312 acne vs 3,690 non-acne (8% vs 92%).
A naive model predicting non-acne always scores 92.2% accuracy.
F1 (acne class) and AUROC are the meaningful metrics.

**Metrics:** Accuracy, F1 (acne class), AUROC

**Prerequisites:** Run `07_train_classifier.ipynb` and download DermNet to `data/dermnet/`.

In [ ]:
import os
from pathlib import Path

if Path('/content').exists():
    os.chdir('/content/AcneDetection')
print(f'Working directory: {os.getcwd()}')

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
from pathlib import Path
from PIL import Image

%matplotlib inline

DERMNET_DIR = Path('data/dermnet')
CLF_OUT     = Path('outputs/classifier')
OUT_PART2   = Path('outputs/part2')
OUT_PART2.mkdir(parents=True, exist_ok=True)
ACNE_FOLDER = 'Acne and Rosacea Photos'
CONF        = 0.5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load DermNet test set

In [ ]:
class DermNetBinary(Dataset):
    """DermNet with binary acne / non-acne labels."""
    def __init__(self, split, transform):
        self.samples = []
        root = DERMNET_DIR / split
        for folder in sorted(root.iterdir()):
            label = 1 if folder.name == ACNE_FOLDER else 0
            for img_path in sorted(folder.glob('*')):
                if img_path.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                    self.samples.append((img_path, label))
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return self.transform(Image.open(path).convert('RGB')), label

test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_ds     = DermNetBinary('test', test_tf)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False,
                         num_workers=2, pin_memory=True)

acne_count = sum(1 for _, l in test_ds.samples if l == 1)
print(f'Test set : {len(test_ds)} images')
print(f'  acne={acne_count}  non_acne={len(test_ds)-acne_count}')
print(f'  Naive baseline accuracy: {1 - acne_count/len(test_ds):.4f}')

## 2. Load model

In [ ]:
model = models.efficientnet_b0(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(model.classifier[1].in_features, 2),
)
model.load_state_dict(torch.load(str(CLF_OUT / 'best.pth'),
                                  map_location=device, weights_only=False))
model.to(device).eval()
print('Model loaded from outputs/classifier/best.pth')

## 3. Run inference & compute metrics

In [ ]:
all_probs, all_preds, all_labels = [], [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs  = imgs.to(device)
        probs = torch.softmax(model(imgs), dim=1)[:, 1]
        preds = (probs >= CONF).long()
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs  = np.array(all_probs)
all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc   = accuracy_score(all_labels, all_preds)
f1    = f1_score(all_labels, all_preds, pos_label=1, zero_division=0)
auroc = roc_auc_score(all_labels, all_probs)

print(f'Accuracy : {acc:.4f}  (naive baseline: {1 - acne_count/len(test_ds):.4f})')
print(f'F1 (acne): {f1:.4f}')
print(f'AUROC    : {auroc:.4f}')

results = {
    'naive_baseline': {'accuracy': 1 - acne_count / len(test_ds)},
    'baseline': {'accuracy': acc, 'f1_acne': f1, 'auroc': auroc},
}
out_path = Path('outputs/part2') / 'dermnet_results.json'
Path('outputs/part2').mkdir(parents=True, exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved -> {out_path}')  # Run 08_domain_adaptation.ipynb for full ablation

## 4. Confusion matrix & ROC curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Layout: rows=Predicted, cols=Actually, positive (acne) first
# sklearn cm: rows=actual, cols=predicted → transpose + flip for target layout
cm = confusion_matrix(all_labels, all_preds)
cm_display = cm.T[::-1, ::-1]
disp = ConfusionMatrixDisplay(cm_display, display_labels=['acne', 'non-acne'])
disp.plot(ax=axes[0], colorbar=False)
axes[0].set_xlabel('Actually')
axes[0].set_ylabel('Predicted')
axes[0].set_title('Confusion Matrix — DermNet Test Set')

fpr, tpr, _ = roc_curve(all_labels, all_probs)
axes[1].plot(fpr, tpr, label=f'AUROC = {auroc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — DermNet Test Set')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_PART2 / 'dermnet_evaluation.png', dpi=150)
plt.show()
print('Saved -> outputs/part2/dermnet_evaluation.png')